<div style="background:#1a4a2e;padding:24px 32px;border-radius:10px;margin-bottom:8px">
<h1 style="color:#ffffff;margin:0 0 6px 0">Urban Tree Canopy Detection with USDA NAIP Imagery</h1>
<p style="color:#b8e0c8;margin:0;font-size:1.05em">A Teaching Module in Open, Replicable and Reproducible GeoAI Workflow — I-GUIDE Platform</p>
</div>

**Author:** Dr. Yi Qi, Spatial Sciences Institute, University of Southern California  
**Dataset:** NAIP aerial imagery over Boyle Heights and City Terrace, Los Angeles  
**Task:** Binary semantic segmentation — classify every pixel as *tree canopy* or *background*  
**Model:** U-Net with a ResNet-34 encoder  
**Platform:** I-GUIDE JupyterHub at [jupyter.i-guide.io](https://jupyter.i-guide.io) (self-contained — all packages and data are installed/downloaded below)

---

### Learning Objectives

By the end of this notebook you will be able to:

1. Explain **semantic segmentation** and how it differs from classification and object detection
2. Describe the **U-Net architecture** and the role of skip connections
3. Load and visualize **GeoTIFF aerial imagery** and binary canopy labels
4. Understand how **data augmentation** helps when labeled data is scarce
5. Read and understand a **training loop** — forward pass, loss, backward pass, optimizer step
6. Load a pre-trained model and evaluate it using **Dice, IoU, Precision, and Recall**
7. Run **inference** and interpret pixel-level error maps (TP / FP / FN)
8. Explore the effect of the **decision threshold** on the Precision/Recall trade-off

---

### How to run this notebook

Execute cells **top to bottom** using **Shift + Enter** (or the ▶ button in the toolbar).

| Symbol | Meaning |
|--------|---------|
| **⚙️** | Contains parameters you are encouraged to change |
| **❓** | Reflection question — write your answer directly in the cell |
| **✅ REQUIRED** | Run this cell — the notebook depends on it |
| **⚠️ OPTIONAL** | Read for understanding; skip running on I-GUIDE (CPU-only) |

> **Tip:** If you see an import error, go to **Kernel → Restart Kernel and Run All Cells**.

---
## Section 0 — Install Dependencies

The I-GUIDE JupyterHub environment may not have all required packages pre-installed.  
Run this cell **once** at the start of each session. It is safe to re-run.

| Package | Purpose |
|---------|---------|
| `torch` / `torchvision` | Deep learning framework (GPU-accelerated) |
| `segmentation-models-pytorch` | Ready-made U-Net with pre-trained encoders |
| `albumentations` | Fast image augmentation library |
| `rasterio` | Read and write GeoTIFF files (preserves CRS metadata) |
| `matplotlib` / `numpy` | Visualisation and numerical computing |

> ⏱ First-time installation takes **2–4 minutes**. Subsequent runs are near-instant (packages are cached by the kernel).

In [ ]:
# Install all required packages
# Using %pip (not !pip) is the recommended practice on JupyterHub —
# it installs into the same kernel that is running this notebook.
#
# Note: the data download in Section 2 uses Python's built-in urllib,
# so no extra packages are needed for that step.

%pip install --quiet \
    torch torchvision torchaudio

%pip install --quiet \
    segmentation-models-pytorch \
    albumentations \
    rasterio \
    pyyaml \
    matplotlib \
    numpy

print("Installation complete.")

> **⚠️ After installation completes, you may see the message:**  
> *"Note: you may need to restart the kernel to use updated packages."*  
>
> If you do, go to **Kernel → Restart Kernel**, then re-run all cells from the top (Sections 0 → 1 → 2 → …).  
> If the installation output ends with `Installation complete.` and no errors appear, you can continue without restarting.

---
## Section 1 — Environment Check

Confirm that all imports succeed and report which device PyTorch will use.

> **I-GUIDE JupyterHub runs on CPU only.** GPU is not required for this tutorial — model training is optional (Section 8), and all inference sections use the pre-trained model that is included in the dataset download.

In [ ]:
import os
import sys
import time
import zipfile
import warnings
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import rasterio
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"  # suppress version-check warning
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

warnings.filterwarnings("ignore", category=UserWarning)

# ── Device selection ───────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    device_info = f"cuda  ({gpu_name}, {gpu_mem:.1f} GB)"
else:
    device      = torch.device("cpu")
    device_info = "cpu  (GPU not available — training is optional, inference runs fine on CPU)"

print(f"Python          : {sys.version.split()[0]}")
print(f"PyTorch         : {torch.__version__}")
print(f"Device in use   : {device_info}")
print(f"albumentations  : {A.__version__}")
print(f"rasterio        : {rasterio.__version__}")
print(f"smp             : {smp.__version__}")
print()
print("✅  All packages imported successfully.")

---
## Section 2 — Download Dataset and Pre-trained Model

All data for this tutorial is published on the **I-GUIDE Platform Data Portal** and downloaded directly using Python's built-in `urllib` library — no extra packages required.

The single zip archive (`iguide.zip`) contains **everything** you need:

| Contents | Description |
|----------|-------------|
| `unet_dataset/train/` | 510 labelled aerial tiles for training |
| `unet_dataset/val/`   | 27 tiles for validation during training |
| `unet_dataset/test/`  | 150 tiles held out for final evaluation |
| `pretrained/`         | Weights from a fully-trained model for comparison |

> **Citation:** Yoo, J., Qi, Y., et al. (2025). *Urban Tree Canopy Data for Deep Learning Applications in Boyle Heights and City Terrace, Los Angeles, California, USA*. I-GUIDE Platform. https://doi.org/10.17605/OSF.IO/IGUIDE
>
> ⏱ Download is ~300–400 MB and takes **2–5 minutes** depending on network speed.  
> The cell is **idempotent** — it skips the download if the data is already present.

In [ ]:
# ── Download data from the I-GUIDE Platform Data Portal ───────────────────────
#
# urlretrieve is part of Python's standard library (no pip install needed).
# The zip is published at a stable DOI-backed URL on I-GUIDE storage.
# ─────────────────────────────────────────────────────────────────────────────

from urllib.request import urlretrieve
import zipfile, os, sys

# I-GUIDE Platform dataset URL (published, stable)
DATA_URL  = "https://storage.i-guide.io/datasets/4cb74a02-87e8-4857-8d60-395973248f59/iguide.zip"
ZIP_PATH  = "iguide.zip"
DATA_ROOT = "iguide"          # top-level folder created inside the zip


# ── Simple download progress hook ─────────────────────────────────────────────
def _progress(block_num, block_size, total_size):
    """Print at 0/20/40/60/80/100% to avoid IOPub rate limit."""
    done = block_num * block_size
    if total_size > 0:
        pct = min(done * 100 / total_size, 100)
        if int(pct) % 20 == 0 and int(pct) != getattr(_progress, "_last", -1):
            _progress._last = int(pct)
            print(f"  {int(pct):3d}%  ({done/1e6:.0f}/{total_size/1e6:.0f} MB)")


# ── Download (skipped if already done) ────────────────────────────────────────
if os.path.isdir(DATA_ROOT):
    print(f"✅  Data already available in '{DATA_ROOT}/'")
    print("   (Delete this folder and re-run to re-download.)")
else:
    if not os.path.isfile(ZIP_PATH):
        print(f"Downloading dataset from I-GUIDE Platform...")
        print("  ⏱  Please wait — this may take a few minutes.\n")
        urlretrieve(DATA_URL, ZIP_PATH, reporthook=_progress)
        size_mb = os.path.getsize(ZIP_PATH) / 1e6
        print(f"\n  ✅  Download complete  ({size_mb:.0f} MB  →  '{ZIP_PATH}')")
    else:
        print(f"  Zip file found ({os.path.getsize(ZIP_PATH)/1e6:.0f} MB) — skipping download.")

    print("Extracting...")
    try:
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(".")
        print(f"✅  Extraction complete.  Data is in: {os.path.abspath(DATA_ROOT)}/")
    except zipfile.BadZipFile:
        os.remove(ZIP_PATH)
        raise RuntimeError("Downloaded ZIP is corrupt. Re-run this cell to retry.")

print()
print("Folder layout:")
for sub in ["unet_dataset/train", "unet_dataset/val", "unet_dataset/test", "pretrained"]:
    path = os.path.join(DATA_ROOT, sub)
    ok   = "✅" if os.path.isdir(path) else "❌"
    print(f"  {ok}  {DATA_ROOT}/{sub}/")

In [ ]:
# ── Locate the pre-trained model (included in the zip downloaded above) ────────
#
# The iguide.zip already contains pre-trained weights at:
#   iguide/pretrained/best_model.pt
#
# This cell just verifies the file is present and prints some context about
# what the pre-trained model represents so you can compare it to your own
# trained model in Section 13.
# ─────────────────────────────────────────────────────────────────────────────
import json

PRETRAINED_PATH = os.path.join(DATA_ROOT, "pretrained", "best_model.pt")

if os.path.exists(PRETRAINED_PATH):
    size_mb = os.path.getsize(PRETRAINED_PATH) / 1e6
    print(f"✅  Pre-trained model found ({size_mb:.1f} MB)")
    print(f"   Path: {os.path.abspath(PRETRAINED_PATH)}")

    # Load training metadata if available
    run_info_path = os.path.join(DATA_ROOT, "pretrained", "train",
                                 "resnet34_bs8_lr0.001", "run_info.json")
    if os.path.exists(run_info_path):
        with open(run_info_path) as f:
            info = json.load(f)
        print()
        print("  Pre-trained model training configuration:")
        for k, v in info.items():
            print(f"    {k:<18}: {v}")

    best_info_path = os.path.join(DATA_ROOT, "pretrained", "train",
                                  "resnet34_bs8_lr0.001", "models", "best_epoch_info.txt")
    if os.path.exists(best_info_path):
        print()
        print("  Pre-trained model performance:")
        with open(best_info_path) as f:
            for line in f:
                print(f"    {line.rstrip()}")
    model_available = True
else:
    print("❌  Pre-trained model NOT found at:", PRETRAINED_PATH)
    print()
    print("   This usually means Section 2 did not complete successfully.")
    print("   Re-run the download cell above, then re-run this cell.")
    model_available = False

if not model_available:
    print()
    print("ℹ️  Section 13 (pre-trained model comparison) will be skipped.")
    print("   All other sections work fine without the pre-trained model.")

In [ ]:
# ── Resolve dataset split paths ────────────────────────────────────────────────
# DATA_ROOT was set in the download cell above ("iguide").
# All data lives inside that folder after extraction.

DATA_ROOT = "iguide"                                        # top-level folder from zip
UNET_DATA = os.path.join(DATA_ROOT, "unet_dataset")        # train / val / test splits

TRAIN_IMG = os.path.join(UNET_DATA, "train", "images")
TRAIN_LBL = os.path.join(UNET_DATA, "train", "labels")
VAL_IMG   = os.path.join(UNET_DATA, "val",   "images")
VAL_LBL   = os.path.join(UNET_DATA, "val",   "labels")
TEST_IMG  = os.path.join(UNET_DATA, "test",  "images")
TEST_LBL  = os.path.join(UNET_DATA, "test",  "labels")

print("Dataset directories:")
for label, path in [("Train images", TRAIN_IMG), ("Train labels", TRAIN_LBL),
                    ("Val images",   VAL_IMG),   ("Val labels",   VAL_LBL),
                    ("Test images",  TEST_IMG),  ("Test labels",  TEST_LBL)]:
    ok    = os.path.isdir(path)
    # Count only .tif files (ignoring .tfw world files)
    count = len([f for f in os.listdir(path) if f.endswith(".tif")]) if ok else 0
    icon  = "✅" if ok else "❌"
    print(f"  {icon}  {label:<15} : {count:>4} tiles   {path}")

if not os.path.isdir(TRAIN_IMG):
    print()
    print("⚠️  Directories not found — make sure Section 2 completed successfully.")

---
## Background: Why Urban Trees, and Why Deep Learning?

Urban tree canopy (UTC) delivers measurable public-health and climate benefits: shade cools streets by up to 10°C, trees remove particulate matter from the air, root systems absorb stormwater, and street trees are associated with lower stress and better mental health outcomes. Cities track UTC to target planting programs, assess equity (low-income neighborhoods often have less canopy), and measure progress toward climate-resilience goals.

Mapping UTC at city scale traditionally requires human analysts to trace tree crowns in aerial photos — a task that is accurate but slow and expensive. Deep learning can produce canopy maps in minutes from raw imagery at sub-meter resolution, enabling near-real-time monitoring.

---

### What Is Semantic Segmentation?

| Task | Output per image |
|------|------------------|
| **Image classification** | One label: "contains tree" / "no tree" |
| **Object detection** | Bounding boxes: one box per tree crown |
| **Semantic segmentation** | A label **for every pixel**: tree / background |

Segmentation is the right tool when we need to know *exactly where* canopy is, not just *whether* it is present.

---

### The U-Net Architecture

U-Net (Ronneberger et al., 2015) was designed for biomedical image segmentation with limited training data. Its **U-shape** has two paths:

```
Input (320×320×3 CIR)
      │
 ┌────▼────┐  Encoder — repeated downsampling
 │ Conv+Pool│  extracts features; spatial size shrinks
 └────┬────┘
      │  ×4
 ┌────▼────┐
 │Bottleneck│  most abstract representation
 └────┬────┘
      │  ×4
 ┌────▼────┐  Decoder — repeated upsampling
 │UpConv   │◄─── skip connection carries fine detail
 │ Conv×2  │     from the corresponding encoder stage
 └────┬────┘
      │
Output (320×320×1 logit map)
```

**Skip connections** (the horizontal arrows) copy encoder feature maps directly to the decoder. This lets the network recover sharp spatial detail (e.g., tree crown edges) that would otherwise be lost during downsampling.

We use **`segmentation-models-pytorch`** which wraps U-Net with a **ResNet-34** encoder pretrained on ImageNet. The encoder provides strong features; we fine-tune everything on our aerial imagery.

---
## Section 3 — Data Exploration

Each sample is a matched pair:
- **Image** — 320 × 320 px GeoTIFF with 3 bands (R, G, B), 8-bit (0–255), from NAIP aerial survey
- **Label** — single-band GeoTIFF, same size. Pixel = **1** (tree canopy) or **0** (background)

Files share the same name so they can be paired automatically.  
We use **`rasterio`** to read GeoTIFFs because it preserves the geographic coordinate metadata (CRS + affine transform) needed to write spatially referenced output.

In [ ]:
# ── Load one training sample and print its metadata ───────────────────────────
train_files = sorted(f for f in os.listdir(TRAIN_IMG) if f.endswith(".tif"))
START   = 100    # change this to browse different tiles (0 to len(train_files)-1)
sample_file = train_files[START]

with rasterio.open(os.path.join(TRAIN_IMG, sample_file)) as src:
    img_arr = src.read()              # (bands, H, W) — uint8
    crs     = src.crs
    meta    = src.meta

with rasterio.open(os.path.join(TRAIN_LBL, sample_file)) as src:
    lbl_arr = src.read(1)             # (H, W) — uint8

print(f"File            : {sample_file}")
print(f"Image shape     : {img_arr.shape}   (bands × H × W)")
print(f"Label shape     : {lbl_arr.shape}   (H × W)")
print(f"Image dtype     : {img_arr.dtype},  range [{img_arr.min()}, {img_arr.max()}]")
print(f"Label dtype     : {lbl_arr.dtype},  unique values: {np.unique(lbl_arr).tolist()}")
print(f"CRS             : {crs}")
tree_pct = lbl_arr.mean() * 100
print(f"Tree canopy     : {tree_pct:.1f}% of pixels in this tile")

In [ ]:
# ── Visualize: image | label | overlay ────────────────────────────────────────
img_cir = np.transpose(img_arr, (1, 2, 0))   # CHW → HWC for matplotlib

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(img_cir)
axes[0].set_title("CIR Aerial Image", fontsize=12)
axes[0].axis("off")

axes[1].imshow(lbl_arr, cmap="Greens", vmin=0, vmax=1)
axes[1].set_title("Ground-Truth Label\n(green = tree canopy)", fontsize=12)
axes[1].axis("off")

overlay = np.zeros((*lbl_arr.shape, 4), dtype=np.float32)
overlay[lbl_arr == 1] = [0.0, 0.8, 0.0, 0.45]
axes[2].imshow(img_cir)
axes[2].imshow(overlay)
axes[2].set_title("Overlay", fontsize=12)
axes[2].axis("off")
patch = mpatches.Patch(color="lime", alpha=0.6, label="Tree canopy")
axes[2].legend(handles=[patch], loc="lower right", fontsize=9)

plt.suptitle(f"Sample: {sample_file}", fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Browse 6 training tiles ────────────────────────────────────────────────────
N_SHOW  = 6    # number of tiles to display

fig, axes = plt.subplots(2, N_SHOW, figsize=(N_SHOW * 2.8, 5.5))

for col, fname in enumerate(train_files[START:START + N_SHOW]):
    with rasterio.open(os.path.join(TRAIN_IMG, fname)) as s:
        img = np.transpose(s.read(), (1, 2, 0))
    with rasterio.open(os.path.join(TRAIN_LBL, fname)) as s:
        lbl = s.read(1)
    axes[0, col].imshow(img)
    axes[0, col].axis("off")
    axes[0, col].set_title(fname[:14], fontsize=7)
    axes[1, col].imshow(lbl, cmap="Greens", vmin=0, vmax=1)
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Image",  fontsize=10)
axes[1, 0].set_ylabel("Label",  fontsize=10)
plt.suptitle("Training samples — top: aerial image, bottom: tree canopy label", fontsize=11)
plt.tight_layout()
plt.show()

print(f"Total tiles — Train: {len(train_files)}, "
      f"Val: {len([f for f in os.listdir(VAL_IMG) if f.endswith('.tif')])}, "
      f"Test: {len([f for f in os.listdir(TEST_IMG) if f.endswith('.tif')])}")

### ❓ Reflection — Section 3

**Q1.** Look at the overlay. Are there pixels you think are tree canopy that the label does *not* mark as such? Why might that happen?

**Q2.** The label is binary (0/1). What challenges would arise from finer categories (e.g., deciduous, evergreen, palm, shrub)?

**Q3.** The aerial scene was tiled into 320 × 320 px patches. Why tile rather than feed the full scene into the model?

> ✏️ *Your answers:*
>
> Q1.
>
> Q2.
>
> Q3.

---
## Section 4 — Data Augmentation

Deep learning needs large, diverse training sets. Aerial imagery datasets are expensive to label, so we **artificially expand** the training set with random, label-preserving transforms applied *at training time*.

We use **[Albumentations](https://albumentations.ai/)**, which applies the same geometric transform to both image and mask simultaneously:

| Transform | Effect | Why it helps |
|-----------|--------|--------------|
| `HorizontalFlip` | Mirror left–right | Trees have no preferred orientation |
| `RandomRotate90` | Rotate 0°/90°/180°/270° | Rotation invariance |
| `ColorJitter` | Vary brightness / contrast / saturation | Lighting conditions vary |
| `CLAHE` | Adaptive histogram equalisation | Enhance local contrast |
| `RGBShift` | Shift R/G/B channel values | Sensor / season variation |
| `Normalize` | Normalize to ImageNet mean/std | Stable gradient flow |

**Validation and test sets** receive *only* `Normalize` — no augmentation — so metrics reflect true performance on unmodified data.

In [ ]:
# ── Define transforms (identical to the main training script) ─────────────────
NORM = dict(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0)

train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.5),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
    A.RGBShift(r_shift_limit=15, g_shift_limit=5, b_shift_limit=5, p=0.3),
    A.Normalize(**NORM),
    ToTensorV2(),
], additional_targets={"mask": "mask"})

eval_tf = A.Compose([
    A.Normalize(**NORM),
    ToTensorV2(),
], additional_targets={"mask": "mask"})

print("Transforms defined: train_tf (with augmentation) | eval_tf (normalise only)")

In [ ]:
# ── Show the same tile after 5 independent random augmentations ───────────────
with rasterio.open(os.path.join(TRAIN_IMG, sample_file)) as s:
    raw_img = np.transpose(s.read().astype(np.uint8), (1, 2, 0))
with rasterio.open(os.path.join(TRAIN_LBL, sample_file)) as s:
    raw_lbl = s.read(1).astype(np.uint8)

N_AUG = 5
fig, axes = plt.subplots(2, N_AUG + 1, figsize=((N_AUG + 1) * 2.8, 5.5))

axes[0, 0].imshow(raw_img);  axes[0, 0].set_title("Original", fontsize=9);  axes[0, 0].axis("off")
axes[1, 0].imshow(raw_lbl, cmap="Greens", vmin=0, vmax=1);                  axes[1, 0].axis("off")

for col in range(1, N_AUG + 1):
    out     = train_tf(image=raw_img, mask=raw_lbl)
    aug_img = np.clip(out["image"].permute(1, 2, 0).numpy(), 0, 1)
    aug_lbl = out["mask"].numpy()
    axes[0, col].imshow(aug_img);                             axes[0, col].set_title(f"Aug #{col}", fontsize=9)
    axes[0, col].axis("off")
    axes[1, col].imshow(aug_lbl, cmap="Greens", vmin=0, vmax=1); axes[1, col].axis("off")

axes[0, 0].set_ylabel("Image", fontsize=10)
axes[1, 0].set_ylabel("Label", fontsize=10)
plt.suptitle("Same tile — original vs 5 random augmentations\n"
             "(label is transformed identically to the image)", fontsize=11)
plt.tight_layout()
plt.show()

### ❓ Reflection — Section 4

**Q1.** Why must the *same* geometric transform be applied to both the image and its label?

**Q2.** Why are colour/brightness transforms applied to the *image only*, not the label?

**Q3.** Training augmentation is random — two epochs never see the same transformed tile. How does this reduce overfitting?

> ✏️ *Your answers:*
>
> Q1.
>
> Q2.
>
> Q3.

---
## Section 5 — Dataset and DataLoader

PyTorch organises data loading through two key classes:

**`Dataset`** — Knows how to load *one* sample (image + label pair). You subclass it and implement:
- `__len__()` → how many samples exist
- `__getitem__(idx)` → return the sample at index `idx` as tensors

**`DataLoader`** — Wraps a `Dataset`; handles batching, shuffling, and (optionally) parallel I/O via multiple worker processes.

We define a custom `TIFDataset` class that reads paired GeoTIFF files and applies our augmentation pipeline.

> **Note on `num_workers=0`:** Cloud JupyterHub containers (including I-GUIDE) often restrict spawning sub-processes for parallel data loading. Setting `num_workers=0` forces single-threaded loading inside the main process, which avoids `RuntimeError: DataLoader worker exited unexpectedly` errors common in containerised environments.

In [ ]:
class TIFDataset(Dataset):
    """Paired GeoTIFF image/label dataset for binary segmentation."""

    def __init__(self, images_dir: str, labels_dir: str, transform=None):
        self.images_dir  = images_dir
        self.labels_dir  = labels_dir
        self.transform   = transform
        self.image_files = sorted(f for f in os.listdir(images_dir) if f.lower().endswith(".tif"))
        if not self.image_files:
            raise ValueError(f"No .tif files found in {images_dir}")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        fn = self.image_files[idx]

        # Load GeoTIFFs as uint8 NumPy arrays
        with rasterio.open(os.path.join(self.images_dir, fn)) as s:
            img_np = s.read().astype(np.uint8)           # (C, H, W)
        with rasterio.open(os.path.join(self.labels_dir, fn)) as s:
            lbl_np = s.read(1).astype(np.uint8)          # (H, W)

        img_np = np.transpose(img_np, (1, 2, 0))         # → (H, W, C) for Albumentations
        lbl_np = (lbl_np > 0).astype(np.uint8)           # ensure strictly binary

        if self.transform:
            out   = self.transform(image=img_np, mask=lbl_np)
            img_t = out["image"]             # FloatTensor (C, H, W)
            lbl_t = out["mask"].float()      # FloatTensor (H, W)
        else:
            img_t = torch.from_numpy(img_np.transpose(2, 0, 1) / 255.0).float()
            lbl_t = torch.from_numpy(lbl_np).float()

        return img_t, lbl_t


# ── Instantiate with eval transforms for a quick sanity check ─────────────────
train_ds = TIFDataset(TRAIN_IMG, TRAIN_LBL, transform=eval_tf)
val_ds   = TIFDataset(VAL_IMG,   VAL_LBL,   transform=eval_tf)
test_ds  = TIFDataset(TEST_IMG,  TEST_LBL,  transform=eval_tf)

print(f"Samples — train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

img_t, lbl_t = train_ds[0]
print(f"Image tensor shape : {list(img_t.shape)}   dtype: {img_t.dtype}")
print(f"Label tensor shape : {list(lbl_t.shape)}   dtype: {lbl_t.dtype}")
print(f"Image value range  : [{img_t.min():.3f}, {img_t.max():.3f}]")

---
## Section 6 — Model Architecture

We build a **U-Net** using [`segmentation-models-pytorch`](https://github.com/qubvel/segmentation_models.pytorch).

- **Encoder**: ResNet-34 pretrained on ImageNet — produces features at 5 spatial scales
- **Decoder**: 4 upsampling blocks, each receiving a skip connection from the encoder
- **Head**: 1×1 convolution → one output channel (the raw logit for tree / background)

The output is a **logit** (unbounded real number). During training it goes into `BCEWithLogitsLoss` (which fuses sigmoid + BCE for numerical stability). During inference we apply `torch.sigmoid()` and threshold at 0.5.

In [ ]:
# ── Build U-Net ────────────────────────────────────────────────────────────────
model = smp.Unet(
    encoder_name    = "resnet34",   # ImageNet-pretrained CNN backbone
    encoder_weights = "imagenet",
    in_channels     = 3,            # CIR input (NIR-R-G)
    classes         = 1,            # one output channel (binary)
).to(device)

# ── Parameter counts ──────────────────────────────────────────────────────────
total_p   = sum(p.numel() for p in model.parameters())
enc_p     = sum(p.numel() for p in model.encoder.parameters())
dec_p     = (sum(p.numel() for p in model.decoder.parameters())
           + sum(p.numel() for p in model.segmentation_head.parameters()))

print(f"Total parameters     : {total_p:>12,}")
print(f"  Encoder (ResNet-34): {enc_p:>12,}  ← pretrained weights")
print(f"  Decoder + head     : {dec_p:>12,}  ← randomly initialised")

# ── Verify shapes with a dummy forward pass ───────────────────────────────────
model.eval()
with torch.no_grad():
    dummy_in  = torch.zeros(1, 3, 320, 320, device=device)
    dummy_out = model(dummy_in)
print(f"\nInput  : {list(dummy_in.shape)}")
print(f"Output : {list(dummy_out.shape)}  (logits — pass through sigmoid to get probabilities)")

---
## Section 7 — Evaluation Metrics

All metrics are computed from the **per-pixel confusion matrix**:

|  | Predicted **Tree** | Predicted **Background** |
|--|-------------------|-------------------------|
| **Actual Tree** | TP | FN |
| **Actual Background** | FP | TN |

| Metric | Formula | Meaning |
|--------|---------|--------|
| **Precision** | TP / (TP + FP) | Of pixels predicted as tree, how many actually are? |
| **Recall** | TP / (TP + FN) | Of all true tree pixels, how many did we find? |
| **IoU** (Jaccard) | TP / (TP + FP + FN) | Overlap between predicted and actual canopy |
| **Dice** (F1) | 2·TP / (2·TP + FP + FN) | Harmonic mean of Precision and Recall |

For canopy mapping, **high Recall** (few missed trees) is often more important than high Precision, because a missed tree represents a real gap in coverage data.

In [ ]:
def compute_metrics(pred: torch.Tensor, target: torch.Tensor) -> dict:
    """Compute binary segmentation metrics from integer prediction and target tensors."""
    p   = pred.cpu().numpy().flatten()
    t   = target.cpu().numpy().flatten()
    TP  = int(np.sum((p == 1) & (t == 1)))
    TN  = int(np.sum((p == 0) & (t == 0)))
    FP  = int(np.sum((p == 1) & (t == 0)))
    FN  = int(np.sum((p == 0) & (t == 1)))
    eps = 1e-8
    return dict(
        TP=TP, TN=TN, FP=FP, FN=FN,
        precision = TP / (TP + FP + eps),
        recall    = TP / (TP + FN + eps),
        iou       = TP / (TP + FP + FN + eps),
        dice      = 2 * TP / (2 * TP + FP + FN + eps),
    )


# ── Quick demo ────────────────────────────────────────────────────────────────
perfect = compute_metrics(torch.tensor([1, 1, 0, 0]), torch.tensor([1, 1, 0, 0]))
noisy   = compute_metrics(torch.tensor([1, 0, 1, 0]), torch.tensor([1, 1, 0, 0]))

print("Perfect prediction — "
      f"Dice: {perfect['dice']:.2f}  IoU: {perfect['iou']:.2f}  "
      f"P: {perfect['precision']:.2f}  R: {perfect['recall']:.2f}")
print("Noisy  prediction — "
      f"Dice: {noisy['dice']:.2f}  IoU: {noisy['iou']:.2f}  "
      f"P: {noisy['precision']:.2f}  R: {noisy['recall']:.2f}  "
      f"(1 FP, 1 FN)")

---
## Section 8 — Training  *(Optional — read to understand; skip to run)*

<div style="background:#fff3cd;border-left:5px solid #ffc107;padding:14px 18px;border-radius:6px;margin:10px 0">
<b>⚠️ OPTIONAL on I-GUIDE JupyterHub (CPU only)</b><br>
Training this model on CPU takes <b>50–100 minutes</b>. For this tutorial, a fully trained model is already included in the data download and will be loaded automatically in <b>Section 8b</b> below.<br><br>
<b>Recommended workflow:</b><br>
&nbsp;&nbsp;• <b>Read</b> the explanation and code in this section to understand how training works.<br>
&nbsp;&nbsp;• <b>Skip running</b> the code cells (do not execute them).<br>
&nbsp;&nbsp;• <b>Jump to Section 8b</b> to load the pre-trained model and continue with inference.<br><br>
If you have access to a GPU (e.g., running locally or on a GPU-enabled cluster), you can run this section normally — it takes ~4–7 minutes on a modern GPU.
</div>

### Loss function: `BCEWithLogitsLoss`

For binary segmentation we use **Binary Cross-Entropy (BCE)**:

$$\mathcal{L}_i = -\bigl[y_i \log \sigma(\hat{y}_i) + (1-y_i)\log(1-\sigma(\hat{y}_i))\bigr]$$

where $\hat{y}_i$ is the model's raw **logit** output, $\sigma$ is the sigmoid function, and $y_i \in \{0,1\}$ is the ground-truth pixel label.  
`BCEWithLogitsLoss` fuses sigmoid + BCE in a single numerically stable operation (avoids overflow in $e^x$).

### Optimizer: Adam

Adam (Adaptive Moment Estimation) maintains per-parameter learning rates using running estimates of gradient mean and variance. It is a robust default for most deep learning tasks.

### Training loop — one epoch:
1. **Forward pass** — images go through the network → logits
2. **Compute loss** — compare logits with ground-truth labels
3. **Backward pass** (`loss.backward()`) — compute gradients via automatic differentiation
4. **Weight update** (`optimizer.step()`) — adjust all parameters in the direction that reduces loss
5. **Validate** on the val split (no gradient updates — no learning)
6. **Save best model** when val Dice improves

---
### ⚙️ Hyperparameters *(only needed if you choose to run training)*

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ⚠️  OPTIONAL — only run this cell if you have decided to train the model.
#     If you are on I-GUIDE JupyterHub (CPU only), SKIP to Section 8b below.
# ══════════════════════════════════════════════════════════════════════════════

# ⚙️ Hyperparameters
LEARNING_RATE = 0.001    # How fast weights are updated. Try: 0.01, 0.001, 0.0001
BATCH_SIZE    = 8        # Tiles per gradient update. Try: 4, 8 (reduce if out-of-memory)
EPOCHS        = 10       # Full passes through training data. Try: 5, 10, 20
THRESHOLD     = 0.5      # Sigmoid decision threshold (explored further in Section 11)
USE_AUG       = True    # Set False to train without Albumentations augmentation

# ── Estimated wall-clock time ──────────────────────────────────────────────────
#   GPU (A100/V100/A6000) :  ~25–40 s/epoch  →  10 epochs ≈  4–7 min  ✅ feasible
#   CPU only (I-GUIDE)    :  ~5–10 min/epoch →  10 epochs ≈  50–100 min  ❌ too slow

print("Hyperparameters set (training is OPTIONAL — see Section 8 header).")
for k, v in [("Learning rate", LEARNING_RATE), ("Batch size", BATCH_SIZE),
             ("Epochs", EPOCHS), ("Threshold", THRESHOLD),
             ("Augmentation", USE_AUG)]:
    print(f"  {k:<16}: {v}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ⚠️  OPTIONAL — only run this cell if you have decided to train the model.
#     CPU-only estimate: 50–100 minutes for 10 epochs.
#     After this cell finishes (or if you skip it), continue to Section 8b.
# ══════════════════════════════════════════════════════════════════════════════

# ── Build loaders, model, loss, optimizer ─────────────────────────────────────
# num_workers=0 is required on most cloud JupyterHub containers
act_train_tf = train_tf if USE_AUG else eval_tf

train_ds     = TIFDataset(TRAIN_IMG, TRAIN_LBL, transform=act_train_tf)
val_ds       = TIFDataset(VAL_IMG,   VAL_LBL,   transform=eval_tf)
test_ds      = TIFDataset(TEST_IMG,  TEST_LBL,  transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model     = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet",
                     in_channels=3, classes=1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = dict(train_loss=[], val_loss=[],
               val_dice=[], val_iou=[], val_precision=[], val_recall=[])
best_dice    = 0.0
best_weights = None

print(f"Training on: {device}")
print(f"  {len(train_ds)} train | {len(val_ds)} val | {len(test_ds)} test tiles")
print(f"  LR={LEARNING_RATE}  BS={BATCH_SIZE}  Epochs={EPOCHS}  Aug={USE_AUG}")
print("-" * 72)

session_start = time.time()

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, lbls.unsqueeze(1))
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_ds)

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    agg = dict(TP=0, TN=0, FP=0, FN=0)
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits  = model(imgs)
            val_loss += criterion(logits, lbls.unsqueeze(1)).item() * imgs.size(0)
            preds   = (torch.sigmoid(logits) > THRESHOLD).long().squeeze(1)
            m       = compute_metrics(preds, lbls.long())
            for k in agg: agg[k] += m[k]
    val_loss /= len(val_ds)

    eps   = 1e-8
    vP    = agg["TP"] / (agg["TP"] + agg["FP"] + eps)
    vR    = agg["TP"] / (agg["TP"] + agg["FN"] + eps)
    vDice = 2 * agg["TP"] / (2 * agg["TP"] + agg["FP"] + agg["FN"] + eps)
    vIoU  = agg["TP"] / (agg["TP"] + agg["FP"] + agg["FN"] + eps)

    for k, v in [("train_loss", train_loss), ("val_loss", val_loss),
                 ("val_dice", vDice), ("val_iou", vIoU),
                 ("val_precision", vP), ("val_recall", vR)]:
        history[k].append(v)

    flag = ""
    if vDice > best_dice:
        best_dice    = vDice
        best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        flag = "  ◀ best"

    elapsed = time.time() - t0
    print(f"Ep {epoch:>2}/{EPOCHS} "
          f"| loss {train_loss:.4f}/{val_loss:.4f} "
          f"| Dice {vDice:.4f}  IoU {vIoU:.4f} "
          f"| P {vP:.4f}  R {vR:.4f} "
          f"| {elapsed:.0f}s{flag}")

total_time = time.time() - session_start
print("-" * 72)
print(f"Done — best val Dice: {best_dice:.4f}   Total: {total_time/60:.1f} min")

---
## Section 8b — Load Active Model  *(Required — run this regardless of Section 8)*

<div style="background:#d4edda;border-left:5px solid #28a745;padding:14px 18px;border-radius:6px;margin:10px 0">
<b>✅ REQUIRED — everyone runs this cell.</b><br>
This cell automatically detects whether you ran the training section and:<br>
&nbsp;&nbsp;• If you <b>skipped training</b>: loads the pre-trained model included in the dataset.<br>
&nbsp;&nbsp;• If you <b>ran training</b>: restores your best checkpoint from that run.<br><br>
Either way, the variable <code>model</code> is ready for all inference sections below.
</div>

In [ ]:
# ── Section 8b: Load Active Model ─────────────────────────────────────────────
# This cell is REQUIRED and safe to run whether or not you ran Section 8.
# ─────────────────────────────────────────────────────────────────────────────

_student_trained = (
    "best_weights" in globals()
    and best_weights is not None           # training ran and improved
)

if _student_trained:
    # ── Restore the best checkpoint saved during training ─────────────────────
    model.load_state_dict({k: v.to(device) for k, v in best_weights.items()})
    model.eval()
    print("✅  Active model: YOUR trained model")
    print(f"   Best val Dice achieved during training: {best_dice:.4f}")
else:
    # ── Load the pre-trained model bundled with the dataset ───────────────────
    if not os.path.exists(PRETRAINED_PATH):
        raise FileNotFoundError(
            f"Pre-trained model not found: {PRETRAINED_PATH}\n"
            "Make sure Section 2 completed successfully."
        )
    model = smp.Unet(
        encoder_name="resnet34", encoder_weights=None,
        in_channels=3, classes=1
    ).to(device)
    model.load_state_dict(torch.load(PRETRAINED_PATH, map_location=device, weights_only=True))
    model.eval()
    print("✅  Active model: PRE-TRAINED model  (Section 8 training was skipped)")
    print("   This model was trained for 10 epochs (LR=0.001, BS=8) on the same dataset.")

# ── Set THRESHOLD default if Section 8 hyperparams cell was skipped ───────────
if "THRESHOLD" not in globals():
    THRESHOLD = 0.5
    print(f"   Threshold set to default: THRESHOLD = {THRESHOLD}")

# ── Build test DataLoader and criterion if Section 8 was skipped ──────────────
if "test_loader" not in globals():
    for _name, _cell in [("TIFDataset", "Cell 21 (Section 5 — Dataset class)"),
                         ("eval_tf",    "Cell 17 (Section 4 — Transforms)"),
                         ("TEST_IMG",   "Cell  9 (Section 3 — Data paths)")]:
        if _name not in globals():
            raise NameError(
                f"{_name!r} is not defined. "
                f"Please run {_cell} first, then re-run Section 8b."
            )
    test_ds     = TIFDataset(TEST_IMG, TEST_LBL, transform=eval_tf)
    test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=0)
    criterion   = nn.BCEWithLogitsLoss()
    print(f"   Test DataLoader created ({len(test_ds)} tiles, batch_size=8)")
print()
print("Model is ready — continue with Section 9 (evaluation and inference).")

---
## Section 9 — Learning Curves and Test Evaluation

**Learning curves** show how loss and metrics evolve over epochs. They are a key diagnostic tool:

| Pattern | Interpretation |
|---------|---------------|
| Train & val loss both high | **Underfitting** — model too simple, or too few epochs |
| Train loss falls, val loss rises | **Overfitting** — model memorises training data |
| Both losses fall and plateau together | **Good fit** |

> **If you skipped training (Section 8):** the learning-curves plot below is skipped automatically — run Section 13 instead to see the pre-trained model's bundled training curves. The test-set evaluation and all remaining sections work normally with the pre-trained model.

In [ ]:
# ── Learning curves — only plotted if you ran Section 8 training ──────────────
if "history" in globals() and len(history.get("train_loss", [])) > 0:
    ep_x = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(ep_x, history["train_loss"], "o-", ms=4, label="Train")
    axes[0].plot(ep_x, history["val_loss"],   "s-", ms=4, label="Val")
    axes[0].set_title("BCE Loss", fontsize=12); axes[0].set_xlabel("Epoch")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(ep_x, history["val_dice"], "o-", ms=4, color="green",  label="Dice")
    axes[1].plot(ep_x, history["val_iou"],  "s-", ms=4, color="orange", label="IoU")
    axes[1].set_title("Val Dice & IoU", fontsize=12); axes[1].set_xlabel("Epoch")
    axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    axes[2].plot(ep_x, history["val_precision"], "o-", ms=4, color="blue", label="Precision")
    axes[2].plot(ep_x, history["val_recall"],    "s-", ms=4, color="red",  label="Recall")
    axes[2].set_title("Val Precision & Recall", fontsize=12); axes[2].set_xlabel("Epoch")
    axes[2].set_ylim(0, 1); axes[2].legend(); axes[2].grid(True, alpha=0.3)

    _hp = f"LR={LEARNING_RATE}  BS={BATCH_SIZE}  Epochs={len(ep_x)}"
    plt.suptitle(f"Your training history  ({_hp})", fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print("ℹ️  Training was skipped — no learning curves to show.")
    print("   See Section 13 for the pre-trained model's training curves.")

In [ ]:
# ── Evaluate the active model on the held-out test set ────────────────────────
# Works whether you trained your own model or loaded the pre-trained one.
model.eval()
test_loss = 0.0
agg = dict(TP=0, TN=0, FP=0, FN=0)

with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        logits = model(imgs)
        test_loss += criterion(logits, lbls.unsqueeze(1)).item() * imgs.size(0)
        preds = (torch.sigmoid(logits) > THRESHOLD).long().squeeze(1)
        m = compute_metrics(preds, lbls.long())
        for k in agg: agg[k] += m[k]

test_loss /= len(test_ds)
eps   = 1e-8
tP    = agg["TP"] / (agg["TP"] + agg["FP"] + eps)
tR    = agg["TP"] / (agg["TP"] + agg["FN"] + eps)
tDice = 2 * agg["TP"] / (2 * agg["TP"] + agg["FP"] + agg["FN"] + eps)
tIoU  = agg["TP"] / (agg["TP"] + agg["FP"] + agg["FN"] + eps)

_label = "YOUR TRAINED MODEL" if _student_trained else "PRE-TRAINED MODEL"
print("\n" + "═" * 52)
print(f"  TEST SET RESULTS  [{_label}]")
print("═" * 52)
print(f"  Loss      : {test_loss:.4f}")
print(f"  Dice      : {tDice:.4f}")
print(f"  IoU       : {tIoU:.4f}")
print(f"  Precision : {tP:.4f}")
print(f"  Recall    : {tR:.4f}")
print(f"  TP={agg['TP']:,}  FP={agg['FP']:,}  FN={agg['FN']:,}  TN={agg['TN']:,}")
print("═" * 52)

### ❓ Reflection — Section 9

**Q1.** Look at the test results. Is Precision higher or lower than Recall? What does that say about the most common type of error the model makes?

**Q2.** Why do we evaluate on a separate *test* set rather than reporting the best validation Dice as the final result?

**Q3.** *(For students who ran training)* Do your learning curves suggest underfitting, overfitting, or a good fit? What visual feature tells you? If you skipped training, answer this question using the pre-trained curves in Section 13.

> ✏️ *Your answers:*
>
> Q1.
>
> Q2.
>
> Q3.

---
## Section 10 — Prediction Visualisation

Numbers alone don't show *where* the model fails. A 4-column layout lets us see exactly which pixels are correct (TP), false alarms (FP), and missed trees (FN).

In [ ]:
# ── Visualise predictions on N_VIS test tiles ─────────────────────────────────
N_VIS = 4
test_files = sorted(f for f in os.listdir(TEST_IMG) if f.endswith(".tif"))

model.eval()
fig, axes = plt.subplots(N_VIS, 4, figsize=(14, N_VIS * 3.4))

for ax, title in zip(axes[0], ["Aerial Image", "Ground Truth",
                                 "Prediction", "Errors (FP=red  FN=blue)"]):
    ax.set_title(title, fontsize=11, fontweight="bold")

for row, fname in enumerate(test_files[:N_VIS]):
    with rasterio.open(os.path.join(TEST_IMG, fname)) as s:
        raw_img = np.transpose(s.read().astype(np.uint8), (1, 2, 0))
    with rasterio.open(os.path.join(TEST_LBL, fname)) as s:
        gt = s.read(1).astype(np.uint8)

    inp = eval_tf(image=raw_img, mask=gt)["image"].unsqueeze(0).to(device)
    with torch.no_grad():
        prob = torch.sigmoid(model(inp)).squeeze().cpu().numpy()
    pred = (prob > THRESHOLD).astype(np.uint8)

    # Error map  — white=TP  red=FP  blue=FN  black=TN
    err = np.zeros((*gt.shape, 3), dtype=np.uint8)
    err[(pred == 1) & (gt == 1)] = [255, 255, 255]
    err[(pred == 1) & (gt == 0)] = [220,  50,  50]
    err[(pred == 0) & (gt == 1)] = [ 50,  80, 220]

    axes[row, 0].imshow(raw_img)
    axes[row, 1].imshow(gt,   cmap="Greens", vmin=0, vmax=1)
    axes[row, 2].imshow(pred, cmap="Greens", vmin=0, vmax=1)
    axes[row, 3].imshow(err)
    for ax in axes[row]: ax.axis("off")
    axes[row, 0].set_ylabel(fname[:15], fontsize=8)

legend = [
    mpatches.Patch(color="white",   label="TP — correct tree"),
    mpatches.Patch(color="#dc3232", label="FP — false alarm"),
    mpatches.Patch(color="#3250dc", label="FN — missed tree"),
    mpatches.Patch(color="black",   label="TN — correct background"),
]
fig.legend(handles=legend, loc="lower center", ncol=4,
           fontsize=10, frameon=True, bbox_to_anchor=(0.75, -0.01))

plt.suptitle("Test set predictions", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Section 11 — Decision Threshold Analysis

The sigmoid output $\sigma(\hat{y}) \in [0,1]$ is a **confidence score**. Converting it to a binary mask requires a threshold $\tau$:

$$\text{prediction} = \begin{cases} 1 & \text{if } \sigma(\hat{y}) > \tau \\ 0 & \text{otherwise} \end{cases}$$

- **Lower $\tau$** → predict more pixels as tree → higher **Recall**, lower **Precision**
- **Higher $\tau$** → predict fewer pixels as tree → lower **Recall**, higher **Precision**

The right choice depends on your application goals.

In [ ]:
# ── Collect all test-set probability maps (single forward pass) ───────────────
model.eval()
all_probs, all_lbls = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        prob = torch.sigmoid(model(imgs.to(device))).squeeze(1).cpu()
        all_probs.append(prob)
        all_lbls.append(lbls)
all_probs = torch.cat(all_probs)
all_lbls  = torch.cat(all_lbls).long()

# ── Sweep thresholds ─────────────────────────────────────────────────────────
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
records    = []
for thr in thresholds:
    m = compute_metrics((all_probs > thr).long(), all_lbls)
    records.append({"threshold": thr, **m})

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(thresholds, [r["precision"] for r in records], "b-o", ms=5, label="Precision")
axes[0].plot(thresholds, [r["recall"]    for r in records], "r-s", ms=5, label="Recall")
axes[0].axvline(THRESHOLD, color="gray", ls="--", label=f"Current τ={THRESHOLD}")
axes[0].set_title("Precision & Recall vs. Threshold", fontsize=12)
axes[0].set_xlabel("Threshold (τ)"); axes[0].set_ylim(0, 1)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(thresholds, [r["dice"] for r in records], "g-o", ms=5, label="Dice")
axes[1].plot(thresholds, [r["iou"]  for r in records], "m-s", ms=5, label="IoU")
axes[1].axvline(THRESHOLD, color="gray", ls="--", label=f"Current τ={THRESHOLD}")
axes[1].set_title("Dice & IoU vs. Threshold", fontsize=12)
axes[1].set_xlabel("Threshold (τ)"); axes[1].set_ylim(0, 1)
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("Threshold sensitivity — test set", fontsize=11)
plt.tight_layout()
plt.show()

best_r = max(records, key=lambda r: r["dice"])
print(f"Threshold that maximises Dice: {best_r['threshold']}  "
      f"(Dice={best_r['dice']:.4f}  IoU={best_r['iou']:.4f})")

In [ ]:
# ── Show the same tile at five different thresholds ───────────────────────────
with rasterio.open(os.path.join(TEST_IMG, test_files[0])) as s:
    demo_img = np.transpose(s.read().astype(np.uint8), (1, 2, 0))
with rasterio.open(os.path.join(TEST_LBL, test_files[0])) as s:
    demo_lbl = s.read(1).astype(np.uint8)

inp = eval_tf(image=demo_img, mask=demo_lbl)["image"].unsqueeze(0).to(device)
with torch.no_grad():
    prob_map = torch.sigmoid(model(inp)).squeeze().cpu().numpy()

show_thr = [0.2, 0.35, 0.5, 0.65, 0.8]
fig, axes = plt.subplots(1, len(show_thr) + 2, figsize=((len(show_thr) + 2) * 2.8, 3.4))

axes[0].imshow(demo_img);  axes[0].set_title("Image");         axes[0].axis("off")
axes[1].imshow(demo_lbl, cmap="Greens", vmin=0, vmax=1)
axes[1].set_title("Ground Truth"); axes[1].axis("off")

for i, thr in enumerate(show_thr):
    axes[i+2].imshow((prob_map > thr).astype(np.uint8), cmap="Greens", vmin=0, vmax=1)
    axes[i+2].set_title(f"τ = {thr}", fontsize=11)
    axes[i+2].axis("off")

plt.suptitle("Effect of threshold on the binary prediction", fontsize=11)
plt.tight_layout()
plt.show()

### ❓ Reflection — Section 11

**Q1.** At which threshold did you observe the best Dice? Is it exactly 0.5, or different? What does a threshold below/above 0.5 say about the model's calibration?

**Q2.** A city equity team wants to find **all** under-canopied census tracts — missing a tree-poor block matters more than flagging a few extra ones. Should they use a lower or higher threshold? Justify your answer.

**Q3.** A **probability map** (before thresholding) contains richer information than a binary mask. How might you use the probability values directly in a downstream analysis?

> ✏️ *Your answers:*
>
> Q1.
>
> Q2.
>
> Q3.

---
## Section 12 — Hyperparameter Experiments  *(Optional — requires running Section 8)*

<div style="background:#fff3cd;border-left:5px solid #ffc107;padding:12px 16px;border-radius:6px;margin:10px 0">
<b>⚠️ OPTIONAL</b> — this section only applies if you ran the training loop in Section 8 (GPU required for a practical time commitment). If you are on I-GUIDE JupyterHub (CPU only), <b>skip to Section 13</b>.
</div>

**Workflow:** Go back to **Section 8** (the `⚙️` hyperparameter cell), change **one value at a time**, re-run training, then record results below.

### Results Table

| Run | LR | BS | Epochs | Aug | Val Dice | Test Dice | Observation |
|-----|----|----|--------|-----|:--------:|:---------:|-------------|
| 1 — baseline   | 0.001  | 8 | 10 | No  |  |  | Starting point |
| 2 — lower LR   | 0.0001 | 8 | 10 | No  |  |  | Slower convergence? |
| 3 — higher LR  | 0.01   | 8 | 10 | No  |  |  | Unstable loss? |
| 4 — with aug   | 0.001  | 8 | 10 | **Yes** |  |  | Augmentation effect |
| 5 — small BS   | 0.001  | **4** | 10 | No |  |  | Noisier gradients? |
| 6 — more epochs| 0.001  | 8 | **20** | No |  |  | Continued improvement? |
| 7 — your choice| | | | |  |  | |

### Questions to answer after filling the table

1. Which single change had the **largest positive effect** on test Dice?
2. Did augmentation help after 10 epochs? What about 20 epochs?
3. Did a smaller batch size make the loss curve **more or less stable**?
4. Did any run show clear **overfitting** (val loss rising while train loss falls)?

> ✏️ *Your answers:*
>
> 1.
>
> 2.
>
> 3.
>
> 4.

---
## Section 13 — Pre-trained Model: Training History

This section shows the **training curves and performance** of the pre-trained model that was bundled with the dataset download. These are the curves you would see if you ran Section 8 yourself with the default hyperparameters (LR=0.001, BS=8, 10 epochs).

**Studying these curves lets you answer the reflection questions in Section 9 even if you skipped training.**

If you *did* run Section 8, the second part of this section compares your trained model to the pre-trained baseline side-by-side on test tiles.

In [ ]:
# ── Show the pre-trained model's bundled training curves ──────────────────────
plots_dir = os.path.join(DATA_ROOT, "pretrained", "train",
                         "resnet34_bs8_lr0.001", "plots")
curve_files = {
    "Loss (train vs. val)": "loss_curve.png",
    "Dice / F1":            "f1_curve.png",
    "Precision & Recall":   "precision_recall.png",
}
available = {k: os.path.join(plots_dir, v)
             for k, v in curve_files.items()
             if os.path.exists(os.path.join(plots_dir, v))}

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 4))
    if len(available) == 1:
        axes = [axes]
    for ax, (title, path) in zip(axes, available.items()):
        ax.imshow(plt.imread(path))
        ax.set_title(title, fontsize=11)
        ax.axis("off")
    plt.suptitle(
        "Pre-trained model — training curves  (LR=0.001, BS=8, 10 epochs on same dataset)",
        fontsize=10
    )
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️  Bundled training curve images not found in the expected path.")
    print(f"   Expected: {plots_dir}")

# ── Also print the best-epoch summary ─────────────────────────────────────────
best_info_path = os.path.join(DATA_ROOT, "pretrained", "train",
                              "resnet34_bs8_lr0.001", "models", "best_epoch_info.txt")
if os.path.exists(best_info_path):
    print()
    print("Pre-trained model — best epoch summary:")
    with open(best_info_path) as f:
        for line in f:
            print(f"  {line.rstrip()}")

In [ ]:
# ── Side-by-side comparison — only shown if you ran Section 8 training ─────────
if _student_trained:
    print("Comparing your trained model vs. pre-trained model on test tiles...")

    # Load the pre-trained model weights for comparison
    _pt_model = smp.Unet(encoder_name="resnet34", encoder_weights=None,
                         in_channels=3, classes=1).to(device)
    _pt_model.load_state_dict(torch.load(PRETRAINED_PATH, map_location=device, weights_only=True))
    _pt_model.eval()

    # Evaluate pre-trained on test set
    _agg = dict(TP=0, TN=0, FP=0, FN=0)
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = (torch.sigmoid(_pt_model(imgs)) > 0.5).long().squeeze(1)
            m = compute_metrics(preds, lbls.long())
            for k in _agg: _agg[k] += m[k]
    _eps  = 1e-8
    p_dice = 2 * _agg["TP"] / (2 * _agg["TP"] + _agg["FP"] + _agg["FN"] + _eps)
    p_iou  = _agg["TP"] / (_agg["TP"] + _agg["FP"] + _agg["FN"] + _eps)
    print(f"Pre-trained model — Test Dice: {p_dice:.4f}  IoU: {p_iou:.4f}")
    print(f"Your model        — Test Dice: {tDice:.4f}  IoU: {tIoU:.4f}")

    # Visual comparison on 3 test tiles
    test_files = sorted(f for f in os.listdir(TEST_IMG) if f.endswith(".tif"))
    N_CMP = 3
    fig, axes = plt.subplots(N_CMP, 4, figsize=(14, N_CMP * 3.4))
    for ax, t in zip(axes[0], ["Aerial Image", "Ground Truth",
                                f"Your model (Dice≈{tDice:.3f})",
                                f"Pre-trained (Dice≈{p_dice:.3f})"]):
        ax.set_title(t, fontsize=10, fontweight="bold")
    for row, fname in enumerate(test_files[:N_CMP]):
        with rasterio.open(os.path.join(TEST_IMG, fname)) as s:
            raw = np.transpose(s.read().astype(np.uint8), (1, 2, 0))
        with rasterio.open(os.path.join(TEST_LBL, fname)) as s:
            gt  = s.read(1).astype(np.uint8)
        inp = eval_tf(image=raw, mask=gt)["image"].unsqueeze(0).to(device)
        with torch.no_grad():
            p_yours = (torch.sigmoid(model(inp))    > THRESHOLD).squeeze().cpu().numpy().astype(np.uint8)
            p_pt    = (torch.sigmoid(_pt_model(inp)) > 0.5      ).squeeze().cpu().numpy().astype(np.uint8)
        axes[row, 0].imshow(raw)
        axes[row, 1].imshow(gt,      cmap="Greens", vmin=0, vmax=1)
        axes[row, 2].imshow(p_yours, cmap="Greens", vmin=0, vmax=1)
        axes[row, 3].imshow(p_pt,    cmap="Greens", vmin=0, vmax=1)
        for ax in axes[row]: ax.axis("off")
    plt.suptitle("Your trained model vs. pre-trained model — test tiles", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️  Side-by-side comparison is only shown when you run Section 8 training.")
    print("   The training curves above already show the pre-trained model's learning history.")

---
## Section 14 — Save Your Model  *(Optional — only if you ran Section 8 training)*

If you trained your own model in Section 8, save its best checkpoint here so you can reload it in a future session without retraining.

> **I-GUIDE JupyterHub storage note:**  
> Files in your **home directory (`~/`)** persist across JupyterHub sessions — they survive server restarts and are private to your account.  
> The pre-trained model from the dataset download is always available at `iguide/pretrained/best_model.pt` and does not need to be saved separately.

In [ ]:
# Save to the persistent home directory on I-GUIDE JupyterHub
save_dir  = os.path.expanduser("~")          # resolves to /home/jovyan on most JupyterHub setups
save_path = os.path.join(save_dir, "student_unet_model.pt")

if best_weights:
    torch.save(best_weights, save_path)
    print(f"Best model saved to: {save_path}")
    print()
    print("To reload later:")
    print("  import segmentation_models_pytorch as smp, torch")
    print("  model = smp.Unet(encoder_name='resnet34', encoder_weights=None,")
    print("                   in_channels=3, classes=1)")
    print("  model.load_state_dict(torch.load('~/student_unet_model.pt', map_location='cpu', weights_only=True))")
else:
    print("No best_weights found — run Section 8 first.")

---
## Section 15 — Summary and Further Exploration

### What you accomplished in this notebook

| Section | Key concepts covered | Required? |
|---------|---------------------|:---------:|
| **0–1** | Dependency installation; environment check | ✅ |
| **2** | Downloading data from the I-GUIDE Platform via `urlretrieve` | ✅ |
| **3** | GeoTIFF structure; paired image/label tiles; class imbalance | ✅ |
| **4** | Data augmentation; label-preserving geometric transforms | ✅ |
| **5** | PyTorch `Dataset` and `DataLoader`; `num_workers=0` in JupyterHub | ✅ |
| **6** | U-Net architecture; encoder/decoder; skip connections; transfer learning | ✅ |
| **7** | Confusion matrix; Dice, IoU, Precision, Recall | ✅ |
| **8** | BCE loss; Adam optimiser; training loop *(read for understanding)* | Read only |
| **8b** | Loading the active model (pre-trained or your own) | ✅ |
| **9** | Test-set evaluation; learning curves (shown if you trained) | ✅ |
| **10** | Pixel-level error maps (TP / FP / FN visualisation) | ✅ |
| **11** | Sigmoid threshold; Precision–Recall trade-off | ✅ |
| **12** | Systematic hyperparameter experiments | Optional (GPU) |
| **13** | Pre-trained training curves; side-by-side comparison (if you trained) | ✅ |
| **14** | Saving your trained model | Optional (GPU) |

---

### Going further

| Direction | What to try |
|-----------|-------------|
| **Run training** | Try Section 8 on a GPU (local machine, Colab, HPC) to experience training first-hand |
| **Combined loss** | Mix BCE with Dice loss: `smp.losses.DiceLoss` + `BCEWithLogitsLoss` |
| **Focal loss** | `smp.losses.FocalLoss` — down-weights easy negatives to handle class imbalance |
| **Stronger encoder** | Try `resnet50`, `efficientnet-b3`, or `mit_b2` (SegFormer-style) in `smp.Unet` |
| **LR scheduling** | Add `torch.optim.lr_scheduler.CosineAnnealingLR` for a smooth decay over epochs |
| **More augmentation** | Add `A.ElasticTransform`, `A.GridDistortion`, `A.GaussianBlur` |
| **Full-scene mosaic** | Use `rasterio.merge.merge` to stitch tiled predictions into a geo-referenced map |
| **Cross-city transfer** | Fine-tune on imagery from a different city and measure accuracy drop |

---

### ❓ Final Reflection

Write **5–8 sentences** addressing all three prompts below.

1. **Architecture insight:** What role do skip connections play in U-Net, and why would the model perform worse without them?
2. **Real-world concern:** What limitation of this approach concerns you most from a **deployment** perspective — for example: label quality, geographic generalisability, seasonal variation, or class imbalance?
3. **Decision support:** How would you decide whether this canopy map is *accurate enough* to inform a city planning or equity decision? What additional validation would you require?

> ✏️ *Your response:*